In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pytrends.request import TrendReq
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time
from dotenv import load_dotenv
import os 
from itertools import product


In [ ]:
#Fetching inflation data from Eurostat API.

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_manr"

params = {
    "geo": "DE",
    "coicop": "CP00",
    "unit": "RCH_A"
}

r = requests.get(url, params=params)
data = r.json()

time_index = data["dimension"]["time"]["category"]["index"]
values = data["value"]

rows = []

for date, idx in time_index.items():
    value = values.get(str(idx))
    rows.append({
        "month": date,
        "inflation_rate": value
    })

df_inflation = pd.DataFrame(rows)

df_inflation["month"] = pd.to_datetime(df_inflation["month"])

df_inflation = df_inflation.sort_values("month")
df_inflation = df_inflation[(df_inflation["month"] >= "2022-01-01") & (df_inflation["month"] <= "2024-12-31")]
df_inflation.reset_index(drop=True)



In [45]:
df_inflation.head()

,month,inflation_rate
300,2022-01-01,5.1
301,2022-02-01,5.5
302,2022-03-01,7.6
303,2022-04-01,7.8
304,2022-05-01,8.7


In [ ]:
#Fetching Google Trends data for the keywords "Inflation", "Energiekosten" and "Lebenshaltungskosten".

pytrends = TrendReq(hl="de-DE", tz=360)

keywords = ["Inflation", "Energiekosten", "Lebenshaltungskosten"]

dfs = []

for word in keywords:

    pytrends.build_payload([word], timeframe="2022-01-01 2024-12-31", geo="DE")

    df = pytrends.interest_over_time()

    df = df.drop(columns=["isPartial"])

    dfs.append(df)

# merge data 
trends = pd.concat(dfs, axis=1)

trends = trends.reset_index() 


In [ ]:
#Aggreagated to monthly data for better visualization and to match the inflation data
trends["month"] = trends["date"].dt.to_period("M")
trends_monthly = trends.groupby("month").mean(numeric_only=True)
trends_monthly = trends_monthly.reset_index()
trends_monthly.head()


,month,Inflation,Energiekosten,Lebenshaltungskosten
0,2021-12,43.00,8.00,62.0
1,2022-01,61.20,13.20,78.8
2,2022-02,61.50,13.75,77.0
3,2022-03,75.50,47.00,85.0
4,2022-04,73.25,25.00,80.0


In [ ]:
'''
Collecting monthly counts of German-language news articles containing the keywords "Inflation" or "Teuerung" from the GNews API.
Note: The GNews API has a free tier with a limit of 50 requests per day. To avoid hitting the rate limit, we will fetch data month by month and include a delay between requests.
LLM-assisited 
'''


# load API key from env. file
load_dotenv()
API_KEY = os.getenv("API_KEY") 

BASE_URL = "https://gnews.io/api/v4/search"

QUERY = "inflation OR Teuerung"

# Restrict results to german news
COUNTRY = "de"
LANG = "de"

start = datetime(2022, 1, 1)
end   = datetime(2025, 1, 1)  

rows = []
cur = start

# Monthly data collection loop with rate limit handling
while cur < end:
    nxt = cur + relativedelta(months=1)

    params = {
        "q": QUERY,
        "country": COUNTRY,
        "lang": LANG,
        "from": cur.strftime("%Y-%m-%dT00:00:00Z"),
        "to":   (nxt - relativedelta(days=1)).strftime("%Y-%m-%dT23:59:59Z"),
        "max": 1,      
        "page": 1,
        "apikey": API_KEY
    }

    r = requests.get(BASE_URL, params=params, timeout=30)

    if r.status_code == 429:
        print("429 -> warte 60s und versuche den Monat nochmal:", cur.strftime("%Y-%m"))
        time.sleep(60)
        continue

    r.raise_for_status()
    data = r.json()

    total = data.get("totalArticles")
    print("OK", cur.strftime("%Y-%m"), "from/to:", params["from"], params["to"], "news_count:", total)

    rows.append({"month": cur.strftime("%Y-%m-01"), "news_count": total})

    cur = nxt
    time.sleep(1.2)  

news_monthly = pd.DataFrame(rows)
news_monthly["month"] = pd.to_datetime(news_monthly["month"])
news_monthly.head()

OK 2022-01 from/to: 2022-01-01T00:00:00Z 2022-01-31T23:59:59Z news_count: 312
OK 2022-02 from/to: 2022-02-01T00:00:00Z 2022-02-28T23:59:59Z news_count: 267
OK 2022-03 from/to: 2022-03-01T00:00:00Z 2022-03-31T23:59:59Z news_count: 331
OK 2022-04 from/to: 2022-04-01T00:00:00Z 2022-04-30T23:59:59Z news_count: 372
OK 2022-05 from/to: 2022-05-01T00:00:00Z 2022-05-31T23:59:59Z news_count: 566
OK 2022-06 from/to: 2022-06-01T00:00:00Z 2022-06-30T23:59:59Z news_count: 729
OK 2022-07 from/to: 2022-07-01T00:00:00Z 2022-07-31T23:59:59Z news_count: 722
OK 2022-08 from/to: 2022-08-01T00:00:00Z 2022-08-31T23:59:59Z news_count: 649
OK 2022-09 from/to: 2022-09-01T00:00:00Z 2022-09-30T23:59:59Z news_count: 777
OK 2022-10 from/to: 2022-10-01T00:00:00Z 2022-10-31T23:59:59Z news_count: 682
OK 2022-11 from/to: 2022-11-01T00:00:00Z 2022-11-30T23:59:59Z news_count: 733
OK 2022-12 from/to: 2022-12-01T00:00:00Z 2022-12-31T23:59:59Z news_count: 524
OK 2023-01 from/to: 2023-01-01T00:00:00Z 2023-01-31T23:59:59Z ne

,month,news_count
0,2022-01-01,312
1,2022-02-01,267
2,2022-03-01,331
3,2022-04-01,372
4,2022-05-01,566


In [ ]:
"""
Fetch data from Eurostat API and convert it into a flat pandas DataFrame.

Eurostat returns data in a compressed multi-dimensional format:
- 'id' defines the dimension names (e.g., time, geo, coicop)
- 'size' gives the length of each dimension
- 'value' stores data as a flattened index → value mapping

This function reconstructs the full table by mapping flat indices
back to their corresponding dimension labels.
LLM assisted. 
"""

def fetch_eurostat_data(url, value_name):
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    ids = data["id"]                      
    sizes = data["size"]                  
    values = data["value"]                
    dimensions = data["dimension"]

    # Build mapping: dimension -> ordered list of category labels
    # (Eurostat stores categories as index mappings, not ordered lists)
    dim_labels = {}
    for dim in ids:
        cat_index = dimensions[dim]["category"]["index"]
        sorted_labels = [label for label, _ in sorted(cat_index.items(), key=lambda x: x[1])]
        dim_labels[dim] = sorted_labels

    # 
    all_positions = list(product(*[range(s) for s in sizes]))

    rows = []
    for flat_idx_str, val in values.items():
        flat_idx = int(flat_idx_str)
        # Map flat index -> tuple of positions in each dimension
        pos_tuple = all_positions[flat_idx]

        row = {}
        # Translate numeric positions in each dimension to their actual category labels
        for dim_name, pos in zip(ids, pos_tuple):
            row[dim_name] = dim_labels[dim_name][pos]

        row[value_name] = val
        rows.append(row)

    df = pd.DataFrame(rows)

    # Zeitspalte umwandeln
    if "time" in df.columns:
        df["time"] = pd.PeriodIndex(df["time"], freq="M").to_timestamp()

    return df


energy_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
    "prc_hicp_midx?geo=DE&coicop=CP045&unit=I15&sinceTimePeriod=2022&untilTimePeriod=2024"
)

food_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
    "prc_hicp_midx?geo=DE&coicop=CP01&unit=I15&sinceTimePeriod=2022&untilTimePeriod=2024"
)

labour_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
    "une_rt_m?geo=DE&sinceTimePeriod=2022&untilTimePeriod=2024"
)

df_energy = fetch_eurostat_data(energy_url, "energy_price_index")
df_food = fetch_eurostat_data(food_url, "food_price_index")
df_labour = fetch_eurostat_data(labour_url, "unemployment_rate")


In [25]:
#Filtering the labour data 

df_labour_filtered = df_labour[
    (df_labour["s_adj"]=="SA")&
    (df_labour["unit"]=="PC_ACT")&
    (df_labour["age"]=="TOTAL")&
    (df_labour["sex"]=="T")
]
df_labour_filtered
df_labour_filtered = df_labour_filtered.drop(["freq", "s_adj", "age", "unit", "sex", "geo"], axis=1)
df_labour_filtered.head()

,time,unemployment_rate
720,2022-01-01,3.3
721,2022-02-01,3.2
722,2022-03-01,3.2
723,2022-04-01,3.1
724,2022-05-01,3.1


In [26]:
#Filtering the food data

df_food = df_food.drop(["freq", "unit", "coicop", "geo"], axis=1)
df_food.head()

,time,food_price_index
0,2022-01-01,116.8
1,2022-02-01,118.0
2,2022-03-01,119.0
3,2022-04-01,122.8
4,2022-05-01,125.2


In [ ]:
#Filtering the energy data

df_energy.columns
df_energy = df_energy.drop(["freq", "unit", "coicop", "geo"], axis=1)
df_energy.head()

,time,energy_price_index
0,2022-01-01,121.5
1,2022-02-01,124.7
2,2022-03-01,137.9
3,2022-04-01,137.3
4,2022-05-01,141.2


In [ ]:
#Saving the processed data to CSV files in the "processed" folder.

df_energy.to_csv('../data/processed/energy.csv', index=False)
df_food.to_csv('../data/processed/food.csv', index=False) 
df_labour_filtered.to_csv('../data/processed/labour.csv', index=False)
df_inflation.to_csv('../data/processed/inflation.csv', index=False) 
news_monthly.to_csv('../data/processed/news_monthly.csv', index=False)
trends_monthly.to_csv('../data/processed/trends_monthly.csv', index=False)
